In [39]:
email_conversation = """From: 이윤구 (lee36656@naver.com)
To: 김혜선 대리님 (eunchae@teddyinternational.me)
Subject: RAG 솔루션 시연 관련 미팅 제안

안녕하세요, 김혜선 대리님,

저는 포지큐브 이윤구입니다. 최근 귀사에서 AI를 활용한 혁신적인 솔루션을 모색 중이라는 소식을 들었습니다. 포지큐브는 AI 및 RAG 솔루션 분야에서 다양한 경험과 노하우를 가진 기업으로, 귀사의 요구에 맞는 최적의 솔루션을 제공할 수 있다고 자부합니다.

저희 포지큐브의 RAG 솔루션은 귀사의 데이터 활용을 극대화하고, 실시간으로 정확한 정보 제공을 통해 비즈니스 의사결정을 지원하는 데 탁월한 성능을 보입니다. 이 솔루션은 특히 다양한 산업에서의 성공적인 적용 사례를 통해 그 효과를 입증하였습니다.

귀사와의 협력 가능성을 논의하고, 저희 RAG 솔루션의 구체적인 기능과 적용 방안을 시연하기 위해 미팅을 제안드립니다. 다음 주 목요일(7월 18일) 오전 10시에 귀사 사무실에서 만나 뵐 수 있을까요?

미팅 시간을 조율하기 어려우시다면, 편하신 다른 일정을 알려주시면 감사하겠습니다. 김혜선 대리님과의 소중한 만남을 통해 상호 발전적인 논의가 이루어지길 기대합니다.

감사합니다.

이윤구
포지큐브 백엔드팀"""

In [40]:
from langchain_openai import ChatOpenAI
from langchain_core.pydantic_v1 import BaseModel, Field


class EmailSummary(BaseModel):
    person: str = Field(description="메일을 보낸 사람")
    email: str = Field(description="메일을 보낸 사람의 이메일 주소")
    company: str = Field(description="메일을 보낸 사람의 회사")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")
    date: str = Field(description="메일 본문에 언급된 미팅 날짜와 시간")

In [33]:
from langchain_core.output_parsers import PydanticOutputParser

output_parser = PydanticOutputParser(pydantic_object=EmailSummary)

In [41]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """
You are a helpful assistant. Please answer the following questions in KOREAN.

QUESTION:
{question}

EMAIL CONVERSATION:
{email_conversation}

FORMAT:
{format}
"""
)

# format 에 PydanticOutputParser의 부분 포맷팅(partial) 추가
prompt = prompt.partial(format=output_parser.get_format_instructions())

In [42]:
llm = ChatOpenAI(temperature=0, model_name="gpt-4o-mini")

In [36]:
chain = prompt | llm | output_parser

In [43]:
answer = chain.invoke(
    {"email_conversation": email_conversation, "question": "이메일 내용을 분석해줘"}
)

In [44]:
print(answer)

person='이윤구' email='lee36656@naver.com' company='포지큐브' subject='RAG 솔루션 시연 관련 미팅 제안' summary='이윤구가 김혜선 대리님에게 포지큐브의 RAG 솔루션을 소개하고, 미팅을 제안하는 내용의 이메일입니다. 포지큐브는 AI 및 RAG 솔루션 분야에서 경험이 있으며, 귀사의 데이터 활용을 극대화하고 비즈니스 의사결정을 지원할 수 있는 솔루션을 제공할 수 있다고 강조하고 있습니다. 미팅은 7월 18일 오전 10시에 제안되었습니다.' date='2023-07-18'


참고: https://serpapi.com/integrations/python

In [13]:
import os

os.environ["SERPAPI_API_KEY"] = (
    "ad0069e2c4bf1b7160774cd361ba2e9d9de33566b4d5c42769b43cb6554efd1c"
)

In [45]:
from langchain_community.utilities import SerpAPIWrapper

params = {
    "engine": "google",
    "gl": "kr",
    "hl": "ko",
}

search = SerpAPIWrapper(params=params)

In [46]:
print(search.run(f"{answer.person} {answer.company} {answer.email}"))

['생성형 AI 솔루션 리더, 포지큐브는 고객의 가치 성장을 위해 대화형&비전 등 다양한 AI 서비스를 제공합니다.', '포지큐브는 비즈니스를 위한 AI 기술과 서비스를 만듭니다. 단순히 AI 기술로 보여줄 수 있는 새로운 무엇인가를 만들어내거나, 인간을 대체하는 생산성을 위한 도구 ...']


In [50]:
search_result = search.run(f"{answer.person} {answer.company} {answer.email}")
print(search_result)

['생성형 AI 솔루션 리더, 포지큐브는 고객의 가치 성장을 위해 대화형&비전 등 다양한 AI 서비스를 제공합니다.', '포지큐브는 비즈니스를 위한 AI 기술과 서비스를 만듭니다. 단순히 AI 기술로 보여줄 수 있는 새로운 무엇인가를 만들어내거나, 인간을 대체하는 생산성을 위한 도구 ...']


In [51]:
" ".join(eval(search_result))

'생성형 AI 솔루션 리더, 포지큐브는 고객의 가치 성장을 위해 대화형&비전 등 다양한 AI 서비스를 제공합니다. 포지큐브는 비즈니스를 위한 AI 기술과 서비스를 만듭니다. 단순히 AI 기술로 보여줄 수 있는 새로운 무엇인가를 만들어내거나, 인간을 대체하는 생산성을 위한 도구 ...'

In [52]:
# sring -> list 변환
search_result = eval(search_result)

In [53]:
search_result[1]

'포지큐브는 비즈니스를 위한 AI 기술과 서비스를 만듭니다. 단순히 AI 기술로 보여줄 수 있는 새로운 무엇인가를 만들어내거나, 인간을 대체하는 생산성을 위한 도구 ...'